In [ ]:
# Numerical Precision and Hardware Efficiency
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/appendices/a3-precision-performance.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/appendices').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/appendices')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Inspect range and spacing for four floating-point dtypes.

In [ ]:
import math

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
import torch
from torch import nn

# [1]
torch.manual_seed(6050)
torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)

dtype_names = {
    torch.float16: "FP16",
    torch.bfloat16: "BF16",
    torch.float32: "FP32",
    torch.float64: "FP64",
}

# [2]
for dtype in (torch.float16, torch.bfloat16, torch.float32, torch.float64):
    info = torch.finfo(dtype)
    print(
        f"{dtype_names[dtype]:>4}: eps={info.eps:.8e}, "
        f"tiny={info.tiny:.8e}, max={info.max:.8e}"
    )

for dtype in (torch.float16, torch.bfloat16, torch.float32):
    one = torch.tensor(1.0, dtype=dtype)
    next_one = torch.nextafter(one, torch.tensor(float("inf"), dtype=dtype))
    assert next_one - one == torch.finfo(dtype).eps

wide = torch.tensor(70_000.0)
small = torch.tensor(2.0**-26)
print("70,000 stored as FP16 / BF16:",
      wide.to(torch.float16).item(), wide.to(torch.bfloat16).item())
print("2^-26 stored as FP16 / BF16:",
      small.to(torch.float16).item(), small.to(torch.bfloat16).item())

**Plan**

1. Separate rounded-away updates, accumulation loss, and cancellation.

In [ ]:
# [1]
for dtype, update in (
    (torch.float16, 1e-4),
    (torch.bfloat16, 1e-3),
    (torch.float32, 1e-8),
):
    weight = torch.tensor(1.0, dtype=dtype)
    changed = weight - torch.tensor(update, dtype=dtype)
    print(f"{dtype_names[dtype]}: 1 - {update:g} -> {changed.item():g}")
    assert changed == weight

term = torch.tensor(0.125, dtype=torch.float16)
sum16 = torch.tensor(0.0, dtype=torch.float16)
sum32 = torch.tensor(0.0, dtype=torch.float32)
for _ in range(10_000):
    sum16 += term
    sum32 += term.float()

large = torch.tensor(1e8, dtype=torch.float32)
one = torch.tensor(1.0, dtype=torch.float32)
cancel_first = (large - large) + one
small_first = (large + one) - large

print("10,000 serial additions in FP16 / FP32:",
      sum16.item(), sum32.item())
print("cancellation orders:", cancel_first.item(), small_first.item())

**Plan**

1. Contrast stable softmax with overflow and prior rounding.

In [ ]:
# [1]
for dtype in (torch.float16, torch.bfloat16, torch.float32):
    logits = torch.tensor([1000.0, 1001.0, 1002.0], dtype=dtype)
    exponentials = torch.exp(logits)
    naive = exponentials / exponentials.sum()
    stable = torch.softmax(logits, dim=0)
    print(f"{dtype_names[dtype]} stored logits: {logits.tolist()}")
    print("  naive finite:", bool(torch.isfinite(naive).all()),
          "stable:", [round(x, 4) for x in stable.float().tolist()])

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Expose loss scaling and operation-specific CPU autocast dtypes.

In [ ]:
# [1]
tiny_gradient = torch.tensor(2.0**-26, dtype=torch.float32)
scale = 2.0**12
direct = tiny_gradient.to(torch.float16).to(torch.float32)
recovered = (
    (tiny_gradient * scale).to(torch.float16).to(torch.float32) / scale
)
assert direct == 0 and recovered == tiny_gradient
print("gradient direct / scaled-then-unscaled:",
      direct.item(), recovered.item())

torch.manual_seed(6050)
layer = nn.Linear(4, 2, dtype=torch.float32)
inputs = torch.randn(3, 4, dtype=torch.float32)
with torch.autocast("cpu", dtype=torch.bfloat16):
    outputs = layer(inputs)
# [2]
outputs.float().square().mean().backward()

print("input, parameter, output, parameter-gradient dtypes:")
print(inputs.dtype, layer.weight.dtype, outputs.dtype, layer.weight.grad.dtype)

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Build and interrogate a synthetic Roofline.

In [ ]:
# [1]
peak_tflops = 100.0
bandwidth_tbps = 2.0
ridge = peak_tflops / bandwidth_tbps

intensities = torch.logspace(-1, 3, 500)
bandwidth_roof = bandwidth_tbps * intensities
combined_roof = torch.minimum(
    bandwidth_roof, torch.full_like(bandwidth_roof, peak_tflops)
)

examples = {
    "elementwise-style": 0.25,
    "reuse growing": 8.0,
    "compute roof": 80.0,
}
# [2]
for name, intensity in examples.items():
    bound = min(peak_tflops, bandwidth_tbps * intensity)
    print(f"{name:>17}: I={intensity:g}, bound={bound:g} TFLOP/s")
print(f"ridge point: {ridge:g} FLOP/byte")

fig, ax = plt.subplots(figsize=(7.6, 4.6))
ax.loglog(intensities, bandwidth_roof, color="#E57200", lw=2.2,
          ls="--", label="bandwidth roof: 2 TB/s")
ax.loglog(intensities, torch.full_like(intensities, peak_tflops),
          color="#232D4B", lw=2.2, ls=":", label="compute roof: 100 TFLOP/s")
ax.loglog(intensities, combined_roof, color="#2E7D32", lw=3.0,
          label="attainable upper bound")
ax.axvline(ridge, color="#722F37", lw=1.3, alpha=0.8)
ax.text(ridge * 1.12, 0.17, "ridge = 50 FLOP/byte",
        color="#722F37", rotation=90, va="bottom")

for index, (name, intensity) in enumerate(examples.items()):
    bound = min(peak_tflops, bandwidth_tbps * intensity)
    ax.scatter(intensity, bound, s=48, color="#232D4B", zorder=4)
    factor = (1.35, 0.72, 0.67)[index]
    ax.annotate(name, (intensity, bound), xytext=(7, 10 * factor),
                textcoords="offset points", fontsize=9)

ax.set_xlim(0.1, 1000)
ax.set_ylim(0.1, 220)
ax.set_xlabel("operational intensity (FLOP/byte)")
ax.set_ylabel("attainable throughput (TFLOP/s)")
ax.grid(True, which="both", color="#D6DCE5", lw=0.6, alpha=0.7)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01),
          ncol=3, frameon=False, fontsize=8.5)
plt.tight_layout()
plt.show()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Verify materialized attention against online blockwise attention.
3. Check the claimed identities, shapes, or invariants.
4. Report or visualize the measured result.

In [ ]:
# [1]
generator = torch.Generator().manual_seed(6050)
N, D_HEAD, D_VALUE, BLOCK = 7, 4, 3, 3
queries = torch.randn(N, D_HEAD, generator=generator)
keys = torch.randn(N, D_HEAD, generator=generator)
values = torch.randn(N, D_VALUE, generator=generator)

scores = queries @ keys.T / math.sqrt(D_HEAD)
dense_output = torch.softmax(scores, dim=-1) @ values

streamed_rows = []
# [2]
for query in queries:
    running_max = torch.tensor(float("-inf"))
    running_sum = torch.tensor(0.0)
    running_output = torch.zeros(D_VALUE)
    for start in range(0, N, BLOCK):
        stop = min(start + BLOCK, N)
        block_scores = query @ keys[start:stop].T / math.sqrt(D_HEAD)
        new_max = torch.maximum(running_max, block_scores.max())
        old_scale = torch.exp(running_max - new_max)
        block_weights = torch.exp(block_scores - new_max)
        running_sum = old_scale * running_sum + block_weights.sum()
        running_output = (
            old_scale * running_output + block_weights @ values[start:stop]
        )
        running_max = new_max
    streamed_rows.append(running_output / running_sum)

streamed_output = torch.stack(streamed_rows)
gap = (dense_output - streamed_output).abs().max().item()
# [3]
assert torch.allclose(dense_output, streamed_output, rtol=0, atol=1e-12)
# [4]
print(f"dense/online maximum gap: {gap:.3e}")
print(f"full score grid: {N*N} entries; one-row block: at most {BLOCK}")